In [1]:
import json
import joblib
import pandas as pd
import sys
from datetime import datetime, timezone
from pathlib import Path

from sklearn.inspection import permutation_importance
from sklearn.metrics import fbeta_score, make_scorer

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

from exoplanet_detector.config import ARTIFACTS_DIR, DEFAULT_RUN_TAG, get_run_artifact_dirs
from exoplanet_detector.features.feature_selection import FINAL_FEATURE_COLUMNS

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
RUN_TAG = DEFAULT_RUN_TAG
run_dirs = get_run_artifact_dirs(RUN_TAG, create=True)

DEPLOYMENT_ARTIFACT_DIR = run_dirs["deployment"]
DEPLOY_MODELS_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_models.joblib"
DEPLOY_MANIFEST_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_manifest.csv"

FEATURE_ANALYSIS_ARTIFACT_DIR = ARTIFACTS_DIR / "feature_analysis" / RUN_TAG
FEATURE_ANALYSIS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PERMUTATION_IMPORTANCE_PATH = FEATURE_ANALYSIS_ARTIFACT_DIR / "permutation_importance.csv"
FEATURE_ANALYSIS_META_PATH = FEATURE_ANALYSIS_ARTIFACT_DIR / "feature_analysis_meta.json"
FEATURE_IMPORTANCE_MATRIX_PATH = FEATURE_ANALYSIS_ARTIFACT_DIR / "feature_importance_matrix.csv"

deploy_bundle = joblib.load(DEPLOY_MODELS_PATH)
deploy_manifest = pd.read_csv(DEPLOY_MANIFEST_PATH)

deployed_models = {
    deploy_id: {
        "model": payload["model"],
        "model_name": payload["model_name"],
        "profile": payload["profile"],
        "threshold": float(payload["threshold"]),
    }
    for deploy_id, payload in deploy_bundle.items()
}

deployed_model_registry_df = pd.DataFrame(
    [
        {
            "deploy_id": deploy_id,
            "model_name": spec["model_name"],
            "profile": spec["profile"],
            "threshold": spec["threshold"],
        }
        for deploy_id, spec in deployed_models.items()
    ]
).sort_values("deploy_id").reset_index(drop=True)

print(f"Feature-analysis artifacts: {FEATURE_ANALYSIS_ARTIFACT_DIR}")
deployed_model_registry_df

Feature-analysis artifacts: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1


,deploy_id,model_name,profile,threshold
0,deploy_f2,knn,f2,0.363636
1,deploy_precision,hgb,precision_constrained,0.885432
2,deploy_recall,rf,recall_constrained,0.029039


In [3]:
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
missing_final_features = [
    column for column in FINAL_FEATURE_COLUMNS
    if column not in KOI_test_set.columns or column not in K2P_set.columns
]
if missing_final_features:
    missing_csv = ", ".join(missing_final_features)
    raise KeyError(f"Missing FINAL_FEATURE_COLUMNS in test datasets: {missing_csv}")

X_koi_test = KOI_test_set.loc[:, FINAL_FEATURE_COLUMNS].copy()
y_koi_test = KOI_test_set["label"].copy()
groups_koi_test = KOI_test_set["group_id"].copy()

X_k2p = K2P_set.loc[:, FINAL_FEATURE_COLUMNS].copy()
y_k2p = K2P_set["label"].copy()
groups_k2p = K2P_set["group_id"].copy()

X_combined = pd.concat([X_koi_test, X_k2p], axis=0, ignore_index=True)
y_combined = pd.concat([y_koi_test, y_k2p], axis=0, ignore_index=True)
groups_combined = pd.concat([groups_koi_test, groups_k2p], axis=0, ignore_index=True)

evaluation_sets = {
    "KOI_test": (X_koi_test, y_koi_test),
    "K2P": (X_k2p, y_k2p),
    "KOI_test_plus_K2P": (X_combined, y_combined),
}

In [5]:
FORCE_RECOMPUTE_FEATURE_ANALYSIS = False
N_REPEATS = 20
PERMUTATION_RANDOM_STATE = 42
PERMUTATION_N_JOBS = 1
PERMUTATION_SCORING_NAME = "f2"
PERMUTATION_SCORER = make_scorer(fbeta_score, beta=2, zero_division=0)

feature_analysis_meta = {}
config_matches_cache = False
if PERMUTATION_IMPORTANCE_PATH.exists() and FEATURE_ANALYSIS_META_PATH.exists() and not FORCE_RECOMPUTE_FEATURE_ANALYSIS:
    feature_analysis_meta = json.loads(FEATURE_ANALYSIS_META_PATH.read_text())
    config_matches_cache = (
        feature_analysis_meta.get("run_tag") == RUN_TAG
        and feature_analysis_meta.get("scoring") == PERMUTATION_SCORING_NAME
        and feature_analysis_meta.get("n_repeats") == N_REPEATS
        and feature_analysis_meta.get("random_state") == PERMUTATION_RANDOM_STATE
        and feature_analysis_meta.get("n_jobs") == PERMUTATION_N_JOBS
        and set(feature_analysis_meta.get("datasets", [])) == set(evaluation_sets.keys())
    )

if config_matches_cache:
    permutation_importance_df = pd.read_csv(PERMUTATION_IMPORTANCE_PATH)
    print(f"Loaded cached permutation importance: {PERMUTATION_IMPORTANCE_PATH}")
else:
    rows = []
    for deploy_id, spec in deployed_models.items():
        model = spec["model"]
        model_name = spec["model_name"]
        profile = spec["profile"]
        threshold = float(spec["threshold"])

        for dataset_name, (x_eval, y_eval) in evaluation_sets.items():
            result = permutation_importance(
                model,
                x_eval,
                y_eval,
                scoring=PERMUTATION_SCORER,
                n_repeats=N_REPEATS,
                random_state=PERMUTATION_RANDOM_STATE,
                n_jobs=PERMUTATION_N_JOBS,
            )

            dataset_df = pd.DataFrame(
                {
                    "deploy_id": deploy_id,
                    "model_name": model_name,
                    "profile": profile,
                    "threshold": threshold,
                    "dataset": dataset_name,
                    "feature": list(x_eval.columns),
                    "importance_mean": result.importances_mean,
                    "importance_std": result.importances_std,
                    "n_repeats": N_REPEATS,
                    "scoring": PERMUTATION_SCORING_NAME,
                }
            )
            dataset_df["importance_rank"] = (
                dataset_df["importance_mean"].rank(method="dense", ascending=False).astype(int)
            )
            rows.append(dataset_df)

    permutation_importance_df = pd.concat(rows, axis=0, ignore_index=True)
    permutation_importance_df = permutation_importance_df.sort_values(
        ["deploy_id", "dataset", "importance_rank", "feature"],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    permutation_importance_df.to_csv(PERMUTATION_IMPORTANCE_PATH, index=False)

    feature_analysis_meta = {
        "run_tag": RUN_TAG,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "scoring": PERMUTATION_SCORING_NAME,
        "n_repeats": N_REPEATS,
        "random_state": PERMUTATION_RANDOM_STATE,
        "n_jobs": PERMUTATION_N_JOBS,
        "n_deployed_models": len(deployed_models),
        "datasets": list(evaluation_sets.keys()),
        "rows": int(permutation_importance_df.shape[0]),
    }
    FEATURE_ANALYSIS_META_PATH.write_text(json.dumps(feature_analysis_meta, indent=2))
    print(f"Saved permutation importance: {PERMUTATION_IMPORTANCE_PATH}")
    print(f"Saved metadata: {FEATURE_ANALYSIS_META_PATH}")

permutation_importance_df

Loaded cached permutation importance: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1/permutation_importance.csv


,deploy_id,model_name,profile,threshold,dataset,feature,importance_mean,importance_std,n_repeats,scoring,importance_rank
0,deploy_f2,knn,f2,0.363636,K2P,orbital_period_days,0.039938,0.007426,20,f2,1
1,deploy_f2,knn,f2,0.363636,K2P,transit_duration_hours,0.030641,0.004361,20,f2,2
2,deploy_f2,knn,f2,0.363636,K2P,transit_depth,0.017684,0.006941,20,f2,3
3,deploy_f2,knn,f2,0.363636,K2P,a_over_rs,0.016489,0.004517,20,f2,4
4,deploy_f2,knn,f2,0.363636,K2P,stellar_metallicity_dex,0.016095,0.004451,20,f2,5
...,...,...,...,...,...,...,...,...,...,...,...
94,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,impact_parameter,0.014248,0.000994,20,f2,7
95,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,stellar_logg_cgs,0.006482,0.000531,20,f2,8
96,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,stellar_teff_k,0.006087,0.000643,20,f2,9
97,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,transit_duration_hours,0.004770,0.000824,20,f2,10


In [6]:
importance_matrix_long = permutation_importance_df.loc[
    :, ["feature", "deploy_id", "dataset", "importance_mean"]
].copy()
importance_matrix_long["model_dataset"] = (
    importance_matrix_long["deploy_id"] + "__" + importance_matrix_long["dataset"]
)

ordered_model_dataset_columns = [
    f"{deploy_id}__{dataset_name}"
    for deploy_id in deployed_models.keys()
    for dataset_name in evaluation_sets.keys()
]

feature_importance_matrix_df = (
    importance_matrix_long.pivot_table(
        index="feature",
        columns="model_dataset",
        values="importance_mean",
        aggfunc="first",
    )
    .reindex(index=FINAL_FEATURE_COLUMNS)
    .reindex(columns=ordered_model_dataset_columns)
    .reset_index()
)

feature_importance_matrix_df.to_csv(FEATURE_IMPORTANCE_MATRIX_PATH, index=False)
print(f"Saved feature importance matrix: {FEATURE_IMPORTANCE_MATRIX_PATH}")
print(f"Matrix shape: {feature_importance_matrix_df.shape}")
feature_importance_matrix_df

Saved feature importance matrix: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1/feature_importance_matrix.csv
Matrix shape: (11, 10)


model_dataset,feature,deploy_f2__KOI_test,deploy_f2__K2P,deploy_f2__KOI_test_plus_K2P,deploy_recall__KOI_test,deploy_recall__K2P,deploy_recall__KOI_test_plus_K2P,deploy_precision__KOI_test,deploy_precision__K2P,deploy_precision__KOI_test_plus_K2P
0,orbital_period_days,0.074617,0.039938,0.072628,0.035122,0.000195,0.019047,0.103809,0.030832,0.071124
1,impact_parameter,0.019767,0.005691,0.001050,0.032403,-0.000440,0.014248,0.040011,0.008760,0.025768
2,transit_duration_hours,0.058593,0.030641,0.062038,0.009296,-0.000088,0.004770,0.084444,0.017087,0.046779
3,transit_depth,0.053896,0.017684,0.031936,0.062791,-0.002150,0.034220,0.138935,0.055926,0.086622
4,inclination_deg,0.022984,-0.000823,0.007216,0.050961,0.000442,0.027773,0.046802,0.002585,0.024084
5,a_over_rs,0.032001,0.016489,0.024727,0.048842,-0.000817,0.026464,0.210004,0.044286,0.131463
6,planet_radius_rearth,0.076522,0.002708,0.038951,0.068097,-0.000021,0.037464,0.165748,0.006537,0.084659
7,insolation_earth,0.030800,0.004623,0.019758,0.038444,-0.000036,0.018085,0.077360,0.001053,0.039635
8,stellar_teff_k,0.014325,-0.001042,0.009430,0.010156,-0.001233,0.006087,0.083105,0.035272,0.066172
9,stellar_logg_cgs,0.020333,-0.005813,0.005920,0.011963,-0.001130,0.006482,0.080990,0.051070,0.068631
